In [ ]:
get_doc_layers_len <- function(doc) {
    doc$with_read(function(t) { doc$layers$len(t) })
}

get_doc_sources_len <- function(doc) {
    doc$with_read(function(t) { doc$sources$len(t) })
}

get_layer_opacity <- function(doc, layer) {
    doc$with_read(function(t) { doc$layers$get(t, layer)$parameters$opacity })
}

get_layer_name <- function(doc, layer) {
    doc$with_read(function(t) { doc$layers$get(t, layer)$name })
}

In [ ]:
# Must display the doc for any loading the file information.
# This will not run in nbconvert tests so we need to assume no data will be
# loaded from the file.
doc <- jupytergis::GISDocument$new("../data/default.jGIS")
doc

In [ ]:
# This will not run in nbconvert tests so we need to assume no data will be
# loaded from the file.
layers_count <- get_doc_layers_len(doc)
stopifnot(get_doc_layers_len(doc) == layers_count)

sources_count <- get_doc_layers_len(doc)
stopifnot(get_doc_sources_len(doc) == sources_count)

In [ ]:
# Capture doc, layers_count, sources_count
test_add_layer_new_source <- function(func, args) {    
    # Dynamically call doc$<func>(...)
    layer <- do.call(doc[[func]], args)
    layers_count <<- layers_count + 1 
    sources_count <<- sources_count + 1

    stopifnot(get_doc_layers_len(doc) == layers_count)
    stopifnot(get_doc_sources_len(doc) == sources_count)

    if ("opacity" %in% names(args)) {
        stopifnot(get_layer_opacity(doc=doc, layer=layer) == args$opacity)
    }

    layer
}

In [ ]:
google_layer <- test_add_layer_new_source(
    func = "add_raster_layer",
    args = list(
        url = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
        name = "Google Satellite",
        attribution = "Google",
        opacity = 0.7
    )
)

In [ ]:
osm_layer <- test_add_layer_new_source(
    func = "add_raster_layer",
    args = list(
        url = "https://tile.openstreetmap.org/{z}/{x}/{y}.png",
        name = "OSM from R"
    )
)

In [ ]:
buildings_layer <- test_add_layer_new_source(
    func = "add_vectortile_layer",
    args = list(
        url = "https://planetarycomputer.microsoft.com/api/data/v1/vector/collections/ms-buildings/tilesets/global-footprints/tiles/{z}/{x}/{y}",
        name = "Buildings",
        attribution = "Planetary Computer",
        opacity = 1
    )
)

In [ ]:
road_layer <- test_add_layer_new_source(
    func = "add_geojson_layer",
    args = list(
        path = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_roads.geojson"
    )
)

stopifnot(get_layer_name(doc=doc, layer=road_layer) == "ne_10m_roads")

In [ ]:
maplibre_layer <- test_add_layer_new_source(
    func = "add_image_layer",
    args = list(
        url = "https://maplibre.org/maplibre-gl-js/docs/assets/radar.gif",
        coordinates = list(
            c(-80.425, 46.437),
            c(-71.516, 46.437),
            c(-71.516, 37.936),
            c(-80.425, 37.936)
        )
    )
)